# Scientific Paper RAG System

A comprehensive RAG pipeline for scientific papers using:
- **LlamaIndex** for orchestration
- **Weaviate** for multimodal vector storage
- **BAAI/bge-m3** for embeddings
- **Gemini 2.5 Pro** for generation
- **Gemini 2.0 Flash** for evaluation
- **Hybrid Fusion Retriever** with Reciprocal Rank Fusion
- **Arize Phoenix** for observability

## 1. Install Dependencies

In [1]:
# # Run this cell first to install all required packages
# !pip install -q \
#     llama-index \
#     llama-index-llms-gemini \
#     llama-index-embeddings-huggingface \
#     llama-index-vector-stores-weaviate \
#     llama-index-retrievers-bm25 \
#     weaviate-client \
#     openinference-instrumentation-llama-index \
#     ragas \
#     pymupdf \
#     pillow \
#     pdf2image \
#     pytesseract \
#     easyocr \
#     tabula-py \
#     pystemmer \
#     python-dotenv \
#     nest-asyncio \
#     sentence-transformers \
#     FlagEmbedding

## 2. Imports and Setup

In [2]:
import os
import io
import re
import asyncio
import warnings
from pathlib import Path
from typing import List, Dict, Any, Optional
from dataclasses import dataclass
import pdfplumber

import nest_asyncio
nest_asyncio.apply()

import numpy as np
import pandas as pd
import fitz  # PyMuPDF
from PIL import Image
from dotenv import load_dotenv
import Stemmer
import tabula 

# LlamaIndex imports
from llama_index.core import (
    Document,
    VectorStoreIndex,
    Settings,
)
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.postprocessor import SentenceTransformerRerank
from llama_index.core.ingestion import IngestionPipeline
from llama_index.core.retrievers import QueryFusionRetriever
from llama_index.retrievers.bm25 import BM25Retriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.evaluation import (
    FaithfulnessEvaluator,
    RelevancyEvaluator,
)

# LLM and Embeddings
from llama_index.llms.gemini import Gemini
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

from google import genai
from google.genai import types

# Suppress warnings
warnings.filterwarnings('ignore')

print("✅ All imports successful!")

/Users/ackshay/.pyenv/versions/3.13.7/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/ackshay/.pyenv/versions/3.13.7/lib/python3.13/site-packages/llama_index/llms/gemini/base.py:21: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


✅ All imports successful!


## 3. Configuration

In [3]:
load_dotenv()

@dataclass
class Config:
    # API Keys
    GOOGLE_API_KEY: str = os.getenv("GOOGLE_API_KEY", "")
    OPENAI_API_KEY: str = os.getenv("OPENAI_API_KEY", "")
    
    LLM_MODEL: str = "models/gemini-2.5-flash"
    JUDGE_LLM_MODEL: str = "gpt-4o-mini"
    EMBEDDING_MODEL: str = "BAAI/bge-m3"
    
    # Chunking
    CHUNK_SIZE: int = 512
    CHUNK_OVERLAP: int = 128
    
    # Retrieval
    TOP_K: int = 5
    VECTOR_WEIGHT: float = 0.9
    BM25_WEIGHT: float = 0.1
    
    # PDF Path
    PDF_PATH: str = "./attention_is_all_you_need.pdf"

config = Config()

if not config.GOOGLE_API_KEY:
    print("⚠️ GOOGLE_API_KEY not found!")
    print("Please set it: export GOOGLE_API_KEY='your-key-here'")
else:
    print(f"✅ API Key loaded: {config.GOOGLE_API_KEY[:10]}...")

if not config.OPENAI_API_KEY:
    print("⚠️ OPENAI_API_KEY not found!")
    print("Get one at: https://platform.openai.com/api-keys")
else:
    print(f"✅ OpenAI API Key: {config.OPENAI_API_KEY[:10]}...")

✅ API Key loaded: AIzaSyBWSx...
✅ OpenAI API Key: sk-proj-_f...


## 4. Initialize Phoenix Tracing (Optional)

In [4]:
# Uncomment to enable Phoenix tracing
# import phoenix as px
# from openinference.instrumentation.llama_index import LlamaIndexInstrumentor
# from phoenix.otel import register

# px.launch_app()
# tracer_provider = register(project_name="scientific-paper-rag")
# LlamaIndexInstrumentor().instrument(tracer_provider=tracer_provider)
# print("✅ Phoenix tracing at: http://localhost:6006")

## 5. Initialize LLMs and Embeddings

In [5]:
# Initialize Gemini LLMs
llm = Gemini(
    model=config.LLM_MODEL,
    api_key=config.GOOGLE_API_KEY,
    temperature=0.0,
    max_tokens=1024,
    context_window=8192,
)

judge_llm = OpenAI(
    api_key=config.OPENAI_API_KEY,
    model=config.JUDGE_LLM_MODEL,
    max_tokens=512,
    temperature=0.0,
)

# Initialize BGE-M3 Embedding Model
embed_model = HuggingFaceEmbedding(
    model_name=config.EMBEDDING_MODEL,
    trust_remote_code=True,
    embed_batch_size=8,
)

# Set global settings
Settings.llm = llm
Settings.embed_model = embed_model
Settings.chunk_size = config.CHUNK_SIZE
Settings.chunk_overlap = config.CHUNK_OVERLAP

print(f"✅ LLM: {config.LLM_MODEL}")
print(f"✅ Judge: {config.JUDGE_LLM_MODEL}")
print(f"✅ Embeddings: {config.EMBEDDING_MODEL}")

✅ LLM: models/gemini-2.5-flash
✅ Judge: gpt-4o-mini
✅ Embeddings: BAAI/bge-m3


## 6. Initialize Weaviate

In [6]:
# No vector store initialization needed - using in-memory
print("✅ Using LlamaIndex in-memory vector store")

✅ Using LlamaIndex in-memory vector store


## 7. Document Processing Functions

In [7]:
def clean_text(text: str) -> str:
    """Clean extracted text from PDFs."""
    text = re.sub(r'\x00', '', text) 
    text = re.sub(r'\f', '', text)
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\\s+\\d+\\s+$', '', text)
    text = re.sub(r'\b(nan|NaN)\b', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\\[\\d+(?:,\\s*\\d+)*\\]', lambda m: m.group(0), text)
    return text.strip()


def extract_text_from_pdf(pdf_path: str) -> List[Document]:
    """Extract text from PDF using PyMuPDF."""
    print("📄 Extracting text...")
    documents = []
    doc = fitz.open(pdf_path)
    
    for page_num in range(len(doc)):
        page = doc[page_num]
        text = page.get_text("text")
        
        if text.strip():
            cleaned = clean_text(text)
            if cleaned:
                documents.append(Document(
                    text=cleaned,
                    metadata={
                        "source": pdf_path,
                        "page": page_num + 1,
                        "type": "text",
                    }
                ))
    
    doc.close()
    print(f"   ✅ {len(documents)} pages extracted")
    return documents


def is_reference_table(text: str) -> bool:
    """
    Check if the extracted table is actually a references/bibliography section.
    Returns True if it looks like references (should be filtered out).
    """
    if not text.strip():
        return False
    
    # Reference patterns (bibliography-specific)
    reference_patterns = [
        r'arXiv preprint',
        r'arXiv:\d+\.\d+',
        r'In Proceedings',
        r'Journal of',
        r'et al\.',
        r'pages \d+[–-]\d+',
        r'In Advances in Neural',
        r'preprint arXiv',
    ]
    
    lines = text.split('\n')
    reference_matches = 0
    total_lines = 0
    
    for line in lines:
        line = line.strip()
        if not line or line == 'nan':
            continue
        total_lines += 1
        
        for pattern in reference_patterns:
            if re.search(pattern, line, re.IGNORECASE):
                reference_matches += 1
                break
    
    # If more than 30% of lines look like references, filter it out
    if total_lines > 0 and reference_matches / total_lines > 0.3:
        return True
    
    return False


def extract_tables_from_pdf(pdf_path: str) -> List[Document]:
    """
    Extract tables using hybrid approach:
    1. Tabula (Lattice mode) - for bordered tables
    2. Tabula (Stream mode) - for borderless tables
    3. PDFPlumber - alternative extraction
    4. PyMuPDF - built-in table detection
    
    Each method may extract different information from the same table.
    Only filters out reference sections.
    """
    print("📊 Extracting tables (hybrid approach)...")
    
    all_table_texts = []
    references_filtered = 0
    
    # ==========================================
    # METHOD 1: TABULA LATTICE
    # ==========================================
    print("   🔷 Tabula Lattice...")
    try:
        tables_lattice = tabula.read_pdf(
            pdf_path,
            pages="all",
            multiple_tables=True,
            lattice=True,
            silent=True
        )
        
        for i, df in enumerate(tables_lattice):
            if not df.empty:
                table_text = df.to_markdown(index=False)
                
                if is_reference_table(table_text):
                    references_filtered += 1
                    continue
                
                if table_text.strip():
                    all_table_texts.append(f"[Tabula-Lattice Table {i+1}]\n{table_text}")
                    
        print(f"      Found {len(tables_lattice)} tables")
    except Exception as e:
        print(f"      Error: {e}")
    
    # ==========================================
    # METHOD 2: TABULA STREAM
    # ==========================================
    print("   🔶 Tabula Stream...")
    try:
        tables_stream = tabula.read_pdf(
            pdf_path,
            pages="all",
            multiple_tables=True,
            stream=True,
            silent=True
        )
        
        for i, df in enumerate(tables_stream):
            if not df.empty:
                table_text = df.to_markdown(index=False)
                
                if is_reference_table(table_text):
                    references_filtered += 1
                    continue
                
                if table_text.strip():
                    all_table_texts.append(f"[Tabula-Stream Table {i+1}]\n{table_text}")
                    
        print(f"      Found {len(tables_stream)} tables")
    except Exception as e:
        print(f"      Error: {e}")
    
    # ==========================================
    # METHOD 3: PDFPLUMBER
    # ==========================================
    print("   🔷 PDFPlumber...")
    try:
        table_count = 0
        with pdfplumber.open(pdf_path) as pdf:
            for page_num, page in enumerate(pdf.pages, 1):
                tables = page.extract_tables()
                for table in tables:
                    if table and len(table) > 0:
                        table_count += 1
                        
                        # Convert to text
                        lines = []
                        for row in table:
                            row_text = " | ".join([str(cell) if cell else "" for cell in row])
                            lines.append(row_text)
                        table_text = "\n".join(lines)
                        
                        if is_reference_table(table_text):
                            references_filtered += 1
                            continue
                        
                        if table_text.strip():
                            all_table_texts.append(f"[PDFPlumber Table {table_count} - Page {page_num}]\n{table_text}")
                            
        print(f"      Found {table_count} tables")
    except Exception as e:
        print(f"      Error: {e}")
    
    # ==========================================
    # METHOD 4: PYMUPDF
    # ==========================================
    print("   🔶 PyMuPDF...")
    try:
        table_count = 0
        doc = fitz.open(pdf_path)
        
        for page_num in range(len(doc)):
            page = doc[page_num]
            tables = page.find_tables()
            
            for table in tables:
                table_data = table.extract()
                
                if not table_data or len(table_data) == 0:
                    continue
                
                table_count += 1
                
                # Convert to text
                lines = []
                for row in table_data:
                    row_text = " | ".join([str(cell) if cell else "" for cell in row])
                    lines.append(row_text)
                table_text = "\n".join(lines)
                
                if is_reference_table(table_text):
                    references_filtered += 1
                    continue
                
                if table_text.strip():
                    all_table_texts.append(f"[PyMuPDF Table {table_count} - Page {page_num + 1}]\n{table_text}")
        
        doc.close()
        print(f"      Found {table_count} tables")
    except Exception as e:
        print(f"      Error: {e}")
    
    # ==========================================
    # CREATE DOCUMENTS
    # ==========================================
    table_docs = []
    for i, text in enumerate(all_table_texts):
        table_docs.append(Document(
            text=text,
            metadata={"type": "table", "table_index": i + 1}
        ))
    
    print(f"   📋 References filtered: {references_filtered}")
    print(f"   ✅ Total: {len(table_docs)} table documents")
    
    return table_docs


def extract_images_from_pdf(pdf_path: str) -> List[Dict]:
    """Extract images from PDF."""
    print("🖼️ Extracting images...")
    images = []
    doc = fitz.open(pdf_path)
    
    for page_num in range(len(doc)):
        page = doc[page_num]
        image_list = page.get_images(full=True)
        
        for img_idx, img in enumerate(image_list):
            try:
                xref = img[0]
                base_image = doc.extract_image(xref)
                image_bytes = base_image["image"]
                
                pil_image = Image.open(io.BytesIO(image_bytes))
                if pil_image.width < 100 or pil_image.height < 100:
                    continue
                
                images.append({
                    "page": page_num + 1,
                    "index": img_idx + 1,
                    "width": pil_image.width,
                    "height": pil_image.height,
                    "image": pil_image,
                    "bytes": image_bytes,
                })
            except Exception as e:
                pass
    
    doc.close()
    print(f"   ✅ {len(images)} images extracted")
    return images


def ocr_images(images: List[Dict]) -> List[Document]:
    """Run OCR on images using EasyOCR."""
    if not images:
        return []
    
    print("🔍 Running OCR...")
    import easyocr
    reader = easyocr.Reader(['en'], gpu=False, verbose=False)
    
    ocr_docs = []
    for img_data in images:
        try:
            image = img_data["image"]
            if image.mode != 'RGB':
                image = image.convert('RGB')
            
            results = reader.readtext(np.array(image))
            text = " ".join([r[1] for r in results])
            
            if text.strip():
                ocr_docs.append(Document(
                    text=f"Figure (page {img_data['page']}): {clean_text(text)}",
                    metadata={"type": "image_ocr", "page": img_data["page"]}
                ))
        except:
            pass
    
    print(f"   ✅ OCR completed for {len(ocr_docs)} images")
    return ocr_docs


def create_image_descriptions(images: List[Dict]) -> List[Document]:
    """Create text descriptions for images using Gemini Vision."""
    if not images:
        return []
    
    print("🎨 Creating image descriptions with Vision AI...")
    
    try:
        vision_client = genai.Client(api_key=config.GOOGLE_API_KEY)
        
        docs = []
        for idx, img_data in enumerate(images, 1):
            try:
                print(f"   Processing {idx}/{len(images)} (page {img_data['page']})...", end=" ")
                
                # Convert PIL Image to bytes
                img_buffer = io.BytesIO()
                img_data["image"].save(img_buffer, format='PNG')
                img_bytes = img_buffer.getvalue()
                
                # Prompt for scientific figure description
                prompt = """Analyze this scientific figure/diagram and provide a detailed description.

Focus on:
1. Figure number and Type of diagram (architecture, flowchart, graph, table, etc.)
2. IMPORTANT: Identify the number of independent diagrams and their positions in the image if multiple are present.
3. Main components and their relationships.
4. Any text labels, equations, or annotations.
5. The technical concept illustrated.

Provide a clear, technical description suitable for a research paper."""
                
                # Call vision model
                response = vision_client.models.generate_content(
                    model='gemini-2.5-flash',
                    contents=[
                        types.Content(
                            role="user",
                            parts=[
                                types.Part.from_bytes(
                                    data=img_bytes,
                                    mime_type="image/png"
                                ),
                                types.Part.from_text(text=prompt)
                            ]
                        )
                    ],
                    config=types.GenerateContentConfig(
                        temperature=0.1,
                        max_output_tokens=1024,
                    )
                )
                
                description = response.text.strip()
                doc_text = f"Figure from page {img_data['page']}: {description}"
                
                docs.append(Document(
                    text=doc_text,
                    metadata={"type": "image_description", "page": img_data["page"]}
                ))
                
                print("✅")
                
            except Exception as e:
                print(f"❌ {e}")
                # Fallback
                docs.append(Document(
                    text=f"Figure from page {img_data['page']}, {img_data['width']}x{img_data['height']}px",
                    metadata={"type": "image_description", "page": img_data["page"]}
                ))
        
        print(f"   ✅ {len(docs)} descriptions created")
        return docs
        
    except Exception as e:
        print(f"   ⚠️ Vision API error: {e}")
        # Fallback to basic descriptions
        docs = []
        for img in images:
            docs.append(Document(
                text=f"Figure from page {img['page']}, {img['width']}x{img['height']}px",
                metadata={"type": "image_description", "page": img["page"]}
            ))
        return docs

## 8. Run Document Collection

In [8]:
def collect_all_documents(pdf_path: str) -> List[Document]:
    """Complete document collection pipeline."""
    print("\n" + "="*60)
    print("DOCUMENT COLLECTION PIPELINE")
    print("="*60)
    
    all_docs = []
    
    # Text extraction
    text_docs = extract_text_from_pdf(pdf_path)
    all_docs.extend(text_docs)
    
    # Table extraction
    table_docs = extract_tables_from_pdf(pdf_path)
    all_docs.extend(table_docs)
    
    # Image extraction
    images = extract_images_from_pdf(pdf_path)
    
    # OCR
    ocr_docs = ocr_images(images)
    all_docs.extend(ocr_docs)
    
    # Image descriptions
    img_desc_docs = create_image_descriptions(images)
    all_docs.extend(img_desc_docs)
    
    print(f"\n📚 Total: {len(all_docs)} documents")
    return all_docs

# Run collection
all_documents = collect_all_documents(config.PDF_PATH)


DOCUMENT COLLECTION PIPELINE
📄 Extracting text...
   ✅ 11 pages extracted
📊 Extracting tables (hybrid approach)...
   🔷 Tabula Lattice...


      Found 2 tables
   🔶 Tabula Stream...
      Found 3 tables
   🔷 PDFPlumber...
      Found 1 tables
   🔶 PyMuPDF...
Consider using the pymupdf_layout package for a greatly improved page layout analysis.
      Found 1 tables
   📋 References filtered: 1
   ✅ Total: 5 table documents
🖼️ Extracting images...
   ✅ 3 images extracted
🔍 Running OCR...
   ✅ OCR completed for 3 images
🎨 Creating image descriptions with Vision AI...
   Processing 1/3 (page 3)... ✅
   Processing 2/3 (page 4)... ✅
   Processing 3/3 (page 4)... ✅
   ✅ 3 descriptions created

📚 Total: 22 documents


## 9. Create Ingestion Pipeline

In [9]:
print("\n" + "="*60)
print("INGESTION PIPELINE")
print("="*60)

# Sentence splitter for scientific text
splitter = SentenceSplitter(
    chunk_size=config.CHUNK_SIZE,
    chunk_overlap=config.CHUNK_OVERLAP,
    paragraph_separator="\n\n",
)

# Run pipeline
pipeline = IngestionPipeline(
    transformations=[splitter, embed_model]
)

nodes = pipeline.run(documents=all_documents, show_progress=True)
print(f"\n✅ Created {len(nodes)} nodes")


INGESTION PIPELINE


Generating embeddings: 100%|██████████| 37/37 [00:11<00:00,  3.28it/s]


✅ Created 37 nodes


## 10. Create Vector Index

In [10]:
print("\n" + "="*60)
print("CREATING VECTOR INDEX")
print("="*60)



index = VectorStoreIndex(
    nodes=nodes,
    embed_model=embed_model,
    show_progress=True,
)

print(f"✅ Index created with {len(nodes)} nodes")


CREATING VECTOR INDEX


Generating embeddings: 0it [00:00, ?it/s]

✅ Index created with 37 nodes


## 11. Create Hybrid Fusion Retriever

In [11]:
print("\n" + "="*60)
print("CREATING RETRIEVERS")
print("="*60)


# Vector retriever
vector_retriever = index.as_retriever(similarity_top_k=10)

# BM25 retriever
bm25_retriever = BM25Retriever.from_defaults(
    nodes=nodes,
    similarity_top_k=10,
    stemmer=Stemmer.Stemmer("english"),
    language="english",
)

# Hybrid Fusion with Reciprocal Rank Fusion
hybrid_retriever = QueryFusionRetriever(
    retrievers=[vector_retriever, bm25_retriever],
    retriever_weights=[config.VECTOR_WEIGHT, config.BM25_WEIGHT],
    similarity_top_k=10,
    num_queries=1,
    mode="reciprocal_rerank",  
    use_async=True,
    verbose=False,
)

print(f"✅ Vector Retriever (weight: {config.VECTOR_WEIGHT})")
print(f"✅ BM25 Retriever (weight: {config.BM25_WEIGHT})")
print(f"✅ Hybrid Fusion with Reciprocal Rank Fusion")


CREATING RETRIEVERS
✅ Vector Retriever (weight: 0.9)
✅ BM25 Retriever (weight: 0.1)
✅ Hybrid Fusion with Reciprocal Rank Fusion


In [12]:
print("\n" + "="*60)
print("INITIALIZING RERANKER")
print("="*60)

# Initialize the cross-encoder reranker for second-stage ranking
reranker = SentenceTransformerRerank(
    model="cross-encoder/ms-marco-MiniLM-L-6-v2",
    top_n=5  # Final number of results after reranking
)

print("✅ Reranker: cross-encoder/ms-marco-MiniLM-L-2-v2 (top_n=5)")


INITIALIZING RERANKER
✅ Reranker: cross-encoder/ms-marco-MiniLM-L-2-v2 (top_n=5)


## 12. Create Query Engines

In [13]:
# Query engines WITH reranking
vector_engine = RetrieverQueryEngine.from_args(
    vector_retriever, 
    llm=llm,
    node_postprocessors=[reranker]  # ADD THIS
)

bm25_engine = RetrieverQueryEngine.from_args(
    bm25_retriever, 
    llm=llm,
    node_postprocessors=[reranker]  # ADD THIS
)

hybrid_engine = RetrieverQueryEngine.from_args(
    hybrid_retriever, 
    llm=llm,
    node_postprocessors=[reranker]  # ADD THIS
)

print("✅ Query engines created with reranking")

✅ Query engines created with reranking


## 13. Evaluation Questions

In [14]:
# Load questions from file
with open("transformer_gold_qa_50.txt", "r") as f:
    eval_questions = [line.strip() for line in f if line.strip()]

print(f"✅ Loaded {len(eval_questions)} questions")
print(f"\n📝 First 5 questions:")
for i, q in enumerate(eval_questions[:5], 1):
    print(f"   {i}. {q}...")

print(f"✅ {len(eval_questions)} evaluation questions")

✅ Loaded 50 questions

📝 First 5 questions:
   1. According to Table 2, what was the estimated training cost in FLOPs for the 'Transformer (big)' model on the English-to-French task?...
   2. Looking at Table 3, what was the development set BLEU score for the base model after 100,000 steps?...
   3. Based on Table 1, which layer type has a 'Maximum Path Length' of O(log_k(n))?...
   4. From Table 2, compare the EN-DE BLEU scores of the 'GNMT RL Ensemble' and the 'Transformer (base model)'....
   5. According to Table 3, row A, how does changing the number of heads (h) to 1 affect the BLEU score compared to the base model?...
✅ 50 evaluation questions


In [15]:
print("\n" + "="*60)
print("GENERATING QA DATASET FOR RETRIEVER EVALUATION")
print("="*60)

from llama_index.core.evaluation import generate_question_context_pairs

# Generate question-context pairs from nodes
qa_dataset = generate_question_context_pairs(
    nodes=nodes,
    llm=llm,
    num_questions_per_chunk=1  # 2 questions per chunk
)

# Check the structure
print(f"✅ Generated QA dataset")
print(f"   Queries: {len(qa_dataset.queries)}")
print(f"   Relevant docs: {len(qa_dataset.relevant_docs)}")

# Show sample - queries is a dict, not a list
if qa_dataset.queries:
    # Get first query ID
    first_query_id = list(qa_dataset.queries.keys())[0]
    first_query_text = qa_dataset.queries[first_query_id]
    
    print(f"\n📝 Sample question:")
    print(f"   ID: {first_query_id}")
    print(f"   Text: {first_query_text}")
    
    # Show relevant docs for this query
    if first_query_id in qa_dataset.relevant_docs:
        relevant_node_ids = qa_dataset.relevant_docs[first_query_id]
        print(f"   Relevant docs: {len(relevant_node_ids)} nodes")
else:
    print("\n⚠️ No questions generated!")
    print("   This might happen if:")
    print("   - Dataset is too small")
    print("   - LLM failed to generate questions")
    print("   - Try reducing num_questions_per_chunk to 1")



GENERATING QA DATASET FOR RETRIEVER EVALUATION


100%|██████████| 37/37 [03:29<00:00,  5.66s/it]

✅ Generated QA dataset
   Queries: 37
   Relevant docs: 37

📝 Sample question:
   ID: 5fb611f4-a62e-4780-b8a6-2c9d31aa7afe
   Text: Based on the provided context, describe the fundamental architectural shift introduced by the Transformer model compared to the dominant sequence transduction models it aims to replace. Specifically, what traditional components does it dispense with, and what mechanism does it solely rely on? Additionally, according to the introduction, name one type of neural network that has been firmly established as state-of-the-art in sequence modeling and transduction problems prior to the Transformer.
   Relevant docs: 1 nodes


## 14. Run Evaluation

In [16]:
from llama_index.core.evaluation import RetrieverEvaluator

# Initialize evaluators
print("\n" + "="*60)
print("INITIALIZING EVALUATORS")
print("="*60)

faithfulness_eval = FaithfulnessEvaluator(llm=judge_llm)
relevancy_eval = RelevancyEvaluator(llm=judge_llm)

# Retriever evaluators for MRR, Hit Rate, Precision, Recall
vector_retriever_eval = RetrieverEvaluator.from_metric_names(
    ["mrr", "hit_rate", "precision", "recall"],
    retriever=vector_retriever
)

bm25_retriever_eval = RetrieverEvaluator.from_metric_names(
    ["mrr", "hit_rate", "precision", "recall"],
    retriever=bm25_retriever
)

hybrid_retriever_eval = RetrieverEvaluator.from_metric_names(
    ["mrr", "hit_rate", "precision", "recall"],
    retriever=hybrid_retriever
)

print("✅ Faithfulness & Relevancy evaluators initialized")
print("✅ Retriever evaluators initialized (MRR, Hit Rate, Precision, Recall)")


# ============================================================================
# PART 1: RETRIEVER METRICS (MRR, Hit Rate, Precision, Recall)
# ============================================================================

async def evaluate_retrievers():
    """Evaluate retrievers on qa_dataset."""
    print("\n" + "="*60)
    print("RETRIEVER METRICS ON QA DATASET")
    print("="*60)
    
    # Evaluate each retriever
    print("\n📊 Evaluating Vector Retriever...")
    vector_results = await vector_retriever_eval.aevaluate_dataset(qa_dataset)
    
    print("📊 Evaluating BM25 Retriever...")
    bm25_results = await bm25_retriever_eval.aevaluate_dataset(qa_dataset)
    
    print("📊 Evaluating Hybrid Retriever...")
    hybrid_results = await hybrid_retriever_eval.aevaluate_dataset(qa_dataset)
    
    # Extract metrics
    def get_avg_metrics(results):
        metric_dicts = [r.metric_vals_dict for r in results]
        df = pd.DataFrame(metric_dicts)
        return df.mean()
    
    vector_metrics = get_avg_metrics(vector_results)
    bm25_metrics = get_avg_metrics(bm25_results)
    hybrid_metrics = get_avg_metrics(hybrid_results)
    
    # Create comparison table
    retriever_comparison = pd.DataFrame({
        "Retriever": ["Vector", "BM25", "Hybrid (RRF)"],
        "MRR": [
            vector_metrics["mrr"],
            bm25_metrics["mrr"],
            hybrid_metrics["mrr"]
        ],
        "Hit Rate": [
            vector_metrics["hit_rate"],
            bm25_metrics["hit_rate"],
            hybrid_metrics["hit_rate"]
        ],
        "Precision": [
            vector_metrics["precision"],
            bm25_metrics["precision"],
            hybrid_metrics["precision"]
        ],
        "Recall": [
            vector_metrics["recall"],
            bm25_metrics["recall"],
            hybrid_metrics["recall"]
        ],
    })
    
    print("\n" + "="*60)
    print("RETRIEVER PERFORMANCE COMPARISON")
    print("="*60)
    print(retriever_comparison.to_string(index=False))
    
    return retriever_comparison


# ============================================================================
# PART 2: FAITHFULNESS & RELEVANCY ON EVAL QUESTIONS
# ============================================================================

async def evaluate_rag_quality():
    """Evaluate faithfulness and relevancy on eval_questions."""
    print("\n" + "="*60)
    print("RAG QUALITY EVALUATION ON EVAL QUESTIONS")
    print("="*60)
    
    results = []
    
    for i, q in enumerate(eval_questions, 1):
        print(f"\n🔍 Q{i}: {q[:50]}...")
        
        # Get responses (async)
        vec_resp = await vector_engine.aquery(q)
        bm25_resp = await bm25_engine.aquery(q)
        hybrid_resp = await hybrid_engine.aquery(q)
        
        # Evaluate (async)
        vec_faith = await faithfulness_eval.aevaluate_response(response=vec_resp)
        bm25_faith = await faithfulness_eval.aevaluate_response(response=bm25_resp)
        hybrid_faith = await faithfulness_eval.aevaluate_response(response=hybrid_resp)
        
        vec_rel = await relevancy_eval.aevaluate_response(query=q, response=vec_resp)
        bm25_rel = await relevancy_eval.aevaluate_response(query=q, response=bm25_resp)
        hybrid_rel = await relevancy_eval.aevaluate_response(query=q, response=hybrid_resp)
        
        results.append({
            "question": q,
            "vector_faith": vec_faith.score,
            "vector_rel": vec_rel.score,
            "bm25_faith": bm25_faith.score,
            "bm25_rel": bm25_rel.score,
            "hybrid_faith": hybrid_faith.score,
            "hybrid_rel": hybrid_rel.score,
            "hybrid_response": str(hybrid_resp.response)[:200],
        })
        
        print(f"   Vector: F={vec_faith.score:.2f} R={vec_rel.score:.2f}")
        print(f"   BM25:   F={bm25_faith.score:.2f} R={bm25_rel.score:.2f}")
        print(f"   Hybrid: F={hybrid_faith.score:.2f} R={hybrid_rel.score:.2f}")
    
    return pd.DataFrame(results)


# ============================================================================
# RUN BOTH EVALUATIONS
# ============================================================================

# Run retriever evaluation
retriever_df = await evaluate_retrievers()

# Run RAG quality evaluation
rag_quality_df = await evaluate_rag_quality()


INITIALIZING EVALUATORS
✅ Faithfulness & Relevancy evaluators initialized
✅ Retriever evaluators initialized (MRR, Hit Rate, Precision, Recall)

RETRIEVER METRICS ON QA DATASET

📊 Evaluating Vector Retriever...
📊 Evaluating BM25 Retriever...
📊 Evaluating Hybrid Retriever...

RETRIEVER PERFORMANCE COMPARISON
   Retriever      MRR  Hit Rate  Precision   Recall
      Vector 0.701609  0.918919   0.091892 0.918919
        BM25 0.547555  0.783784   0.078378 0.783784
Hybrid (RRF) 0.696311  0.918919   0.091892 0.918919

RAG QUALITY EVALUATION ON EVAL QUESTIONS

🔍 Q1: According to Table 2, what was the estimated train...
   Vector: F=1.00 R=1.00
   BM25:   F=1.00 R=1.00
   Hybrid: F=1.00 R=1.00

🔍 Q2: Looking at Table 3, what was the development set B...
   Vector: F=1.00 R=1.00
   BM25:   F=1.00 R=1.00
   Hybrid: F=1.00 R=1.00

🔍 Q3: Based on Table 1, which layer type has a 'Maximum ...
   Vector: F=1.00 R=0.00
   BM25:   F=1.00 R=0.00
   Hybrid: F=1.00 R=1.00

🔍 Q4: From Table 2, compare the

## 15. Results Summary

In [17]:
print("\n" + "="*60)
print("COMPLETE EVALUATION SUMMARY")
print("="*60)

# ==========================================
# RETRIEVER METRICS SUMMARY
# ==========================================
print("\n📊 RETRIEVER METRICS:")
print(retriever_df.to_string(index=False))

# Find best retriever for each metric
print("\n🏆 BEST PERFORMERS:")
for metric in ["MRR", "Hit Rate", "Precision", "Recall"]:
    best_idx = retriever_df[metric].idxmax()
    best_retriever = retriever_df.loc[best_idx, "Retriever"]
    best_score = retriever_df.loc[best_idx, metric]
    print(f"   {metric}: {best_retriever} ({best_score:.4f})")

# ==========================================
# RAG QUALITY SUMMARY
# ==========================================
print("\n📊 RAG QUALITY METRICS:")
quality_summary = {
    "Retriever": ["Vector", "BM25", "Hybrid (RRF)"],
    "Avg Faithfulness": [
        rag_quality_df["vector_faith"].mean(),
        rag_quality_df["bm25_faith"].mean(),
        rag_quality_df["hybrid_faith"].mean(),
    ],
    "Avg Relevancy": [
        rag_quality_df["vector_rel"].mean(),
        rag_quality_df["bm25_rel"].mean(),
        rag_quality_df["hybrid_rel"].mean(),
    ],
}

quality_df = pd.DataFrame(quality_summary)
print(quality_df.to_string(index=False))

# ==========================================
# SAVE RESULTS
# ==========================================
os.makedirs("./results", exist_ok=True)
retriever_df.to_csv("./results/retriever_metrics.csv", index=False)
rag_quality_df.to_csv("./results/rag_quality.csv", index=False)
quality_df.to_csv("./results/summary.csv", index=False)

print("\n✅ All results saved to ./results/")
print("   - retriever_metrics.csv (MRR, Hit Rate, Precision, Recall)")
print("   - rag_quality.csv (Faithfulness, Relevancy per question)")
print("   - summary.csv (Overall comparison)")



COMPLETE EVALUATION SUMMARY

📊 RETRIEVER METRICS:
   Retriever      MRR  Hit Rate  Precision   Recall
      Vector 0.701609  0.918919   0.091892 0.918919
        BM25 0.547555  0.783784   0.078378 0.783784
Hybrid (RRF) 0.696311  0.918919   0.091892 0.918919

🏆 BEST PERFORMERS:
   MRR: Vector (0.7016)
   Hit Rate: Vector (0.9189)
   Precision: Vector (0.0919)
   Recall: Vector (0.9189)

📊 RAG QUALITY METRICS:
   Retriever  Avg Faithfulness  Avg Relevancy
      Vector              0.84           0.88
        BM25              0.82           0.92
Hybrid (RRF)              0.90           0.96

✅ All results saved to ./results/
   - retriever_metrics.csv (MRR, Hit Rate, Precision, Recall)
   - rag_quality.csv (Faithfulness, Relevancy per question)
   - summary.csv (Overall comparison)


## 16. Interactive Query Function

In [18]:
def query(question: str):
    """Query the paper using hybrid retrieval."""
    response = hybrid_engine.query(question)
    
    print(f"\n❓ {question}")
    print(f"\n📄 Answer:\n{response.response}")
    print(f"\n📚 Sources ({len(response.source_nodes)}):")
    for i, node in enumerate(response.source_nodes[:3], 1):
        print(f"   {i}. [{node.metadata.get('type', 'text')}] {node.text[:80]}...")
    
    return response

# Test it
query("What is the key innovation of the Transformer?")


❓ What is the key innovation of the Transformer?

📄 Answer:
The Transformer is the first transduction model to rely entirely on self-attention to compute representations of its input and output, without using sequence-aligned recurrent neural networks (RNNs) or convolution. It achieves this by employing stacked self-attention and point-wise, fully connected layers for both its encoder and decoder.

📚 Sources (5):
   1. [image_description] Figure from page 4: This figure presents a single architectural diagram illustra...
   2. [text] Figure 1: The Transformer - model architecture. wise fully connected feed-forwar...
   3. [text] This makes it more difﬁcult to learn dependencies between distant positions [11]...


Response(response='The Transformer is the first transduction model to rely entirely on self-attention to compute representations of its input and output, without using sequence-aligned recurrent neural networks (RNNs) or convolution. It achieves this by employing stacked self-attention and point-wise, fully connected layers for both its encoder and decoder.', source_nodes=[NodeWithScore(node=TextNode(id_='ff7a0355-d331-4004-a969-9a8f4454b65c', embedding=None, metadata={'type': 'image_description', 'page': 4}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='879563a6-1b37-47c3-81fe-554aab1064f0', node_type='4', metadata={'type': 'image_description', 'page': 4}, hash='e46216e12b14290d1bc44a820b4293d9fcc8d647c4240b985c919ee7f7be0502')}, metadata_template='{key}: {value}', metadata_separator='\n', text='Figure from page 4: This figure presents a single architectural diagram illustrating the **Scaled Dot-

In [19]:
query("Explain the difference between two diagrams in Figure 2.")


❓ Explain the difference between two diagrams in Figure 2.

📄 Answer:
Figure 2 illustrates two distinct attention mechanisms: Scaled Dot-Product Attention and Multi-Head Attention.

Scaled Dot-Product Attention takes a Query (Q), Key (K), and Value (V) as input. It computes the dot product of Q and K, scales the result by the square root of the dimension of the keys, optionally applies masking, and then uses a softmax function to obtain weights. These weights are then applied to V to produce the output. This process represents a single attention function.

Multi-Head Attention, on the other hand, consists of several attention layers operating in parallel. Instead of performing a single attention function, it first linearly projects the queries, keys, and values multiple times using different learned projections. Then, it performs the Scaled Dot-Product Attention function on each of these projected versions in parallel. The outputs from these parallel attention functions are subsequent

Response(response='Figure 2 illustrates two distinct attention mechanisms: Scaled Dot-Product Attention and Multi-Head Attention.\n\nScaled Dot-Product Attention takes a Query (Q), Key (K), and Value (V) as input. It computes the dot product of Q and K, scales the result by the square root of the dimension of the keys, optionally applies masking, and then uses a softmax function to obtain weights. These weights are then applied to V to produce the output. This process represents a single attention function.\n\nMulti-Head Attention, on the other hand, consists of several attention layers operating in parallel. Instead of performing a single attention function, it first linearly projects the queries, keys, and values multiple times using different learned projections. Then, it performs the Scaled Dot-Product Attention function on each of these projected versions in parallel. The outputs from these parallel attention functions are subsequently concatenated and projected once more to yield

In [20]:
query("whats the title of the paper?")


❓ whats the title of the paper?

📄 Answer:
The title of the paper is "Attention Is All You Need".

📚 Sources (5):
   1. [text] the input sequence centered around the respective output position. This would in...
   2. [image_description] Figure from page 4: This figure presents a single architectural diagram illustra...
   3. [text] References [1] Jimmy Lei Ba, Jamie Ryan Kiros, and Geoffrey E Hinton. Layer norm...


Response(response='The title of the paper is "Attention Is All You Need".', source_nodes=[NodeWithScore(node=TextNode(id_='48d7481a-6557-404a-a059-6f748c8cc8e5', embedding=None, metadata={'source': './attention_is_all_you_need.pdf', 'page': 7, 'type': 'text'}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='1831ba8c-16dd-4803-a27b-b11c6374fb22', node_type='4', metadata={'source': './attention_is_all_you_need.pdf', 'page': 7, 'type': 'text'}, hash='305dc9bcb61e136c9024c772d9735720b8179ff502b5970959d185e2ea4d4970'), <NodeRelationship.NEXT: '3'>: RelatedNodeInfo(node_id='6ac78388-54f3-4258-b9a6-789780c0fa7c', node_type='1', metadata={}, hash='d7a29da9dea1887d81218ad5d8ecf2d707809dfa038a53aab7267625fe8900b4')}, metadata_template='{key}: {value}', metadata_separator='\n', text='the input sequence centered around the respective output position. This would increase the maximum path length to O(n/r). We pla

In [21]:
# More queries
query("What BLEU score did Transformer achieve on English-German translation?")


❓ What BLEU score did Transformer achieve on English-German translation?

📄 Answer:
The Transformer model achieved a BLEU score of 28.4 on the WMT 2014 English-to-German translation task. The base Transformer model achieved a BLEU score of 27.3 on this task.

📚 Sources (5):
   1. [text] Table 2: The Transformer achieves better BLEU scores than previous state-of-the-...
   2. [text] 6 Results 6.1 Machine Translation On the WMT 2014 English-to-German translation ...
   3. [text] Table 3: Variations on the Transformer architecture. Unlisted values are identic...


Response(response='The Transformer model achieved a BLEU score of 28.4 on the WMT 2014 English-to-German translation task. The base Transformer model achieved a BLEU score of 27.3 on this task.', source_nodes=[NodeWithScore(node=TextNode(id_='dc60f1a3-fa9a-4ff0-9064-1cf30c7f6de3', embedding=None, metadata={'source': './attention_is_all_you_need.pdf', 'page': 8, 'type': 'text'}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='dced43a5-4638-4bfb-86e7-ad49820e3b6f', node_type='4', metadata={'source': './attention_is_all_you_need.pdf', 'page': 8, 'type': 'text'}, hash='1a5717e12a3a527eed9e7bc6f9d46cd627672cc2190866947d266a14265503b9'), <NodeRelationship.NEXT: '3'>: RelatedNodeInfo(node_id='99539562-7701-4f81-ab70-20e070991f78', node_type='1', metadata={}, hash='e939fb121c717a5533baf5898e778ae38d25a60549ec451c6ba592575e564bb2')}, metadata_template='{key}: {value}', metadata_separator='\n', text='Table 2:

In [22]:
query("whats the complexity per layer of self-attention vs recurrent layers?")


❓ whats the complexity per layer of self-attention vs recurrent layers?

📄 Answer:
The complexity per layer for self-attention is O(n^2 · d), while for recurrent layers, it is O(n · d^2).

📚 Sources (5):
   1. [text] We chose the sinusoidal version because it may allow the model to extrapolate to...
   2. [text] Table 1: Maximum path lengths, per-layer complexity and minimum number of sequen...
   3. [text] This makes it more difﬁcult to learn dependencies between distant positions [11]...


Response(response='The complexity per layer for self-attention is O(n^2 · d), while for recurrent layers, it is O(n · d^2).', source_nodes=[NodeWithScore(node=TextNode(id_='c1911ec5-5984-4f9a-a78e-ef0e7a44a145', embedding=None, metadata={'source': './attention_is_all_you_need.pdf', 'page': 6, 'type': 'text'}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='2d4d0b1f-c0e8-40c6-a604-43c57e483f64', node_type='4', metadata={'source': './attention_is_all_you_need.pdf', 'page': 6, 'type': 'text'}, hash='b33fda65640da31b74c0ec39045f359e28f453ced320e75bbc4fcaf5cc4a7ce1'), <NodeRelationship.PREVIOUS: '2'>: RelatedNodeInfo(node_id='395a9012-21b4-489b-97c3-0090b1a36e5c', node_type='1', metadata={'source': './attention_is_all_you_need.pdf', 'page': 6, 'type': 'text'}, hash='36afbf922c8db25fe148633277808b90e4f1861e2c7a3bdcfa56df5e4c29709f')}, metadata_template='{key}: {value}', metadata_separator='\n', text='We c

In [23]:
query("Describe the encoder architecture from Figure 1.")


❓ Describe the encoder architecture from Figure 1.

📄 Answer:
The encoder, depicted in the left half of the model architecture, is comprised of a stack of six identical layers. Each of these layers contains two sub-layers.

The first sub-layer is a multi-head self-attention mechanism. The second sub-layer is a simple, position-wise fully connected feed-forward network.

Residual connections are employed around each of these two sub-layers, and these connections are followed by layer normalization. This means the output of each sub-layer is processed as LayerNorm(x + Sublayer(x)). To support these residual connections, all sub-layers within the model, along with the embedding layers, produce outputs with a dimension of 512.

📚 Sources (5):
   1. [text] Figure 1: The Transformer - model architecture. wise fully connected feed-forwar...
   2. [text] This makes it more difﬁcult to learn dependencies between distant positions [11]...
   3. [image_description] Figure from page 4: This figur

Response(response='The encoder, depicted in the left half of the model architecture, is comprised of a stack of six identical layers. Each of these layers contains two sub-layers.\n\nThe first sub-layer is a multi-head self-attention mechanism. The second sub-layer is a simple, position-wise fully connected feed-forward network.\n\nResidual connections are employed around each of these two sub-layers, and these connections are followed by layer normalization. This means the output of each sub-layer is processed as LayerNorm(x + Sublayer(x)). To support these residual connections, all sub-layers within the model, along with the embedding layers, produce outputs with a dimension of 512.', source_nodes=[NodeWithScore(node=TextNode(id_='2076c3cb-6471-43f7-a5f4-79ca33c099b9', embedding=None, metadata={'source': './attention_is_all_you_need.pdf', 'page': 3, 'type': 'text'}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={<NodeRelationship.SOURCE: '1'>: RelatedN

## 17. Cleanup